In [13]:
import os
os.environ["USE_SYMENGINE"] = "1"

from sympy import *
from sympy.core.backend import *
from delierium.matrix_order import Context, Mgrlex, Mgrevlex, Mlex
from delierium.JanetBasis import Janet_Basis, LHDP
from delierium.helpers import latexer
from IPython.display import Math
from delierium.DerivativeOperators import FrechetD
t=symbols('t')
theta = Function(r'\theta')(t)
print(dir(theta))
latex(theta)
theta._repr_latex_ = "mausi"
theta._repr_latex_()

['__abs__', '__add__', '__annotations__', '__class__', '__complex__', '__delattr__', '__dict__', '__dir__', '__divmod__', '__doc__', '__eq__', '__float__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__int__', '__le__', '__lt__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__pos__', '__pow__', '__radd__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmod__', '__rmul__', '__rpow__', '__rsub__', '__rtruediv__', '__setattr__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__truediv__', '__weakref__', '_diff', '_repr_latex_', '_richcmp_', '_sage_', '_symbolic_', '_sympy_', '_unsafe_reset', 'args', 'args_as_sage', 'args_as_sympy', 'as_coefficients_dict', 'as_numer_denom', 'as_powers_dict', 'as_real_imag', 'atoms', 'coeff', 'copy', 'diff', 'evalf', 'expand', 'free_symbols', 'func', 'get_name', 'has', 'is_Add', 'is_AlgebraicNumber', 'is_A

TypeError: 'str' object is not callable

In [2]:
def prolongation(eq, dependent, independent):
    """

    Doctest stolen from Baumann pp.92/93
    >>> x = var('x')
    >>> u = function('u')
    >>> u_x = u(x)
    >>> f = function("f")
    >>> f_x = f(x, u(x), diff(u(x),x))
    >>> ppp = prolongation([f_x], [u], [x])
    >>> print(ppp[0].expand())
    -D[2](f)(x, u(x), diff(u(x), x))*diff(u(x), x)^2*D[1](xi_1)(x, u(x)) + D[2](f)(x, u(x), diff(u(x), x))*D[1](phi_1)(x, u(x))*diff(u(x), x) - D[2](f)(x, u(x), diff(u(x), x))*diff(u(x), x)*D[0](xi_1)(x, u(x)) + xi_1(x, u(x))*D[0](f)(x, u(x), diff(u(x), x)) + phi_1(x, u(x))*D[1](f)(x, u(x), diff(u(x), x)) + D[2](f)(x, u(x), diff(u(x), x))*D[0](phi_1)(x, u(x))
    >>> # this one here is from Baumann, p.93
    >>> f_x = f(x, u(x), diff(u(x),x),  diff(u(x), x ,x))
    >>> # Baumann's example p. 94
    >>> x = var('x')
    >>> y = function('y')
    >>> print(prolongation([diff(y(x),x,2)], [y], [x])[0].expand())
    -D[1, 1](xi_1)(x, y(x))*diff(y(x), x)^3 + D[1, 1](phi_1)(x, y(x))*diff(y(x), x)^2 - 2*D[0, 1](xi_1)(x, y(x))*diff(y(x), x)^2 - 3*D[1](xi_1)(x, y(x))*diff(y(x), x)*diff(y(x), x, x) + 2*D[0, 1](phi_1)(x, y(x))*diff(y(x), x) - D[0, 0](xi_1)(x, y(x))*diff(y(x), x) + D[1](phi_1)(x, y(x))*diff(y(x), x, x) - 2*D[0](xi_1)(x, y(x))*diff(y(x), x, x) + D[0, 0](phi_1)(x, y(x))
    """
    Depend = [d(*independent) for d in dependent]
    vars = independent + Depend
    xi = [Function("xi_%s" % (j+1), latex_name = r"\xi_{i+1}") for j in range(len(independent))]
    eta = []
    for i in range(len(dependent)):
        phi = Function(f"phi_{i+1}", latex_name = fr"\phi_{i+1}")
        eta.append(phi(*vars) -
                   sum(xi[j](*vars) *
                       Depend[i].diff(independent[j])
                       for j in range(len(independent))))
    test = list(map(lambda _: Function("t_%s" % _),  range(len(Depend))))
    prolong = FrechetD(eq, dependent, independent, testfunction=test)
    prol = []
    for p in prolong:
        _p = []
        for l in p:
            _p.extend([l.subs({test[i], _e}) for _e in eta])
        prol.append(sum(_ for _ in _p))
    prolong = prol[:]
    prol = []
    for j in range(len(prolong)):
        for i in range(len(independent)):
            prol.append(
                (prolong[j] +
                 xi[i](*vars) * sum(_.diff(independent[i]) for _ in eq).expand())
            )
    return prol

In [3]:
x = symbols('x')
u = Function('u')
u_x = u(x)
f = Function("f")
f_x = f(x, u(x), diff(u(x),x))
ppp = prolongation([f_x], [u], [x])

SympifyError: sympy2symengine: Cannot convert '<symengine.lib.symengine_wrapper.UndefFunction object at 0x7f5589677ce0>' (of type <class 'symengine.lib.symengine_wrapper.UndefFunction'>) to a symengine type.